In [2]:
import requests, json, os, pandas as pd
from datetime import datetime


# Paramètres : change ici pour une autre ville
VILLE = "marseille"   # ou "caen"
URL_GBFS = "https://backend.citiz.fr/public/provider/6/gbfs/3.0/gbfs.json"

DOSSIER = f"/lakehouse/default/Files/citiz/citiz_{VILLE}"
os.makedirs(DOSSIER, exist_ok=True)

# 1. Lire le sommaire
gbfs = requests.get(URL_GBFS).json()
feeds = {f["name"]: f["url"] for f in gbfs["data"]["feeds"]}
print("Flux disponibles :", list(feeds.keys()))

# 2. Sauvegarder chaque flux brut avec l'horodatage
horodatage = datetime.now().strftime("%Y-%m-%d_%Hh%M")
for nom, url in feeds.items():
    data = requests.get(url).json()
    with open(f"{DOSSIER}/{nom}_{horodatage}.json", "w") as f:
        json.dump(data, f)
    print("OK :", nom)

# 3. Compter les stations
stations = requests.get(feeds["station_information"]).json()
print(f"\nNombre de stations Citiz à {VILLE} :", len(stations["data"]["stations"]))

# Véhicules dispo par station
status = requests.get(feeds["station_status"]).json()
df_status = pd.json_normalize(status["data"]["stations"])
print("Colonnes station_status :", list(df_status.columns))
print("Voitures dispo en ce moment :", df_status["num_vehicles_available"].sum())

# Liste complète des véhicules
vehicules = requests.get(feeds["vehicle_status"]).json()
df_veh = pd.json_normalize(vehicules["data"]["vehicles"])
print("\nColonnes vehicle_status :", list(df_veh.columns))
print("Nombre total de véhicules :", len(df_veh))

StatementMeta(, 60844337-c8e8-46d4-86b2-0d0277ecba26, 3, Finished, Available, Finished, False)

Flux disponibles : ['geofencing_zones', 'system_pricing_plans', 'gbfs', 'station_information', 'station_status', 'system_information', 'vehicle_status', 'vehicle_types', 'gbfs_versions']
OK : geofencing_zones
OK : system_pricing_plans
OK : gbfs
OK : station_information
OK : station_status
OK : system_information
OK : vehicle_status
OK : vehicle_types
OK : gbfs_versions



Nombre de stations Citiz à marseille : 97
Colonnes station_status : ['station_id', 'num_vehicles_available', 'vehicle_types_available', 'num_docks_available', 'is_installed', 'is_renting', 'is_returning', 'last_reported']
Voitures dispo en ce moment : 193

Colonnes vehicle_status : ['vehicle_id', 'last_reported', 'lat', 'lon', 'is_reserved', 'is_disabled', 'vehicle_type_id', 'current_range_meters', 'current_fuel_percent', 'station_id', 'vehicle_equipment', 'home_station_id']
Nombre total de véhicules : 193
